<a href="https://colab.research.google.com/github/HariDinesh5481/Data-Science/blob/main/jd_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Edu.zip to Edu.zip


In [3]:
import zipfile

In [4]:
with zipfile.ZipFile(
    "/content/Edu.zip",
    "r"
) as zip_ref:

    zip_ref.extractall(
        "resumes"
    )

print("Resume ZIP Extracted")

Resume ZIP Extracted


In [8]:
from pathlib import Path
import pandas as pd
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_path):

    reader = PdfReader(str(pdf_path))

    pages_text = [
        page.extract_text() or ""
        for page in reader.pages
    ]

    return "\n".join(pages_text).strip()


def create_resume_dataframe(folder_path):

    records = []

    folder = Path(folder_path)

    for pdf_file in folder.glob("*.pdf"):

        try:

            records.append({
                "filename": pdf_file.name,
                "text": extract_text_from_pdf(pdf_file)
            })

        except Exception as e:

            records.append({
                "filename": pdf_file.name,
                "text": "",
                "error": str(e)
            })

    return pd.DataFrame(records)

In [9]:
def create_job_dataframe(jd_pdf_path):

    reader = PdfReader(jd_pdf_path)

    records = []

    for page in reader.pages:

        page_text = page.extract_text() or ""

        lines = [
            line.strip()
            for line in page_text.split("\n")
            if line.strip()
        ]

        if not lines:
            continue

        records.append({
            "name": lines[0],
            "text": "\n".join(lines[1:])
        })

    return pd.DataFrame(records)

In [10]:
def save_csv(df, output_path):

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved: {output_path}")

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np

def generate_embeddings(
    texts,
    model_name="sentence-transformers/all-MiniLM-L6-v2"
):

    model = SentenceTransformer(
        model_name
    )

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    return embeddings

In [12]:
def save_embeddings(
    embeddings,
    output_file
):

    np.save(
        output_file,
        embeddings
    )

    print(
        f"Saved: {output_file}"
    )

In [13]:
def load_embeddings(
    embedding_file
):

    embeddings = np.load(
        embedding_file
    )

    return embeddings

In [14]:
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances
)

def calculate_scores(
    job_embeddings,
    resume_embeddings
):

    cosine_scores = cosine_similarity(
        job_embeddings,
        resume_embeddings
    )

    euclidean_scores = euclidean_distances(
        job_embeddings,
        resume_embeddings
    )

    return cosine_scores, euclidean_scores

In [15]:
def get_top_matches(
    job_df,
    resume_df,
    cosine_scores,
    euclidean_scores,
    top_k=10
):

    results = []

    for job_index in range(
        len(job_df)
    ):

        job_name = job_df.loc[
            job_index,
            "name"
        ]

        # Cosine

        top_cosine = np.argsort(
            cosine_scores[job_index]
        )[::-1][:top_k]

        for rank, resume_index in enumerate(
            top_cosine,
            start=1
        ):

            results.append({

                "job_name": job_name,

                "method":
                "Cosine Similarity",

                "rank": rank,

                "resume_name":
                resume_df.loc[
                    resume_index,
                    "filename"
                ],

                "score":
                cosine_scores[
                    job_index
                ][resume_index]
            })

        # Euclidean

        top_euclidean = np.argsort(
            euclidean_scores[job_index]
        )[:top_k]

        for rank, resume_index in enumerate(
            top_euclidean,
            start=1
        ):

            results.append({

                "job_name": job_name,

                "method":
                "Euclidean Distance",

                "rank": rank,

                "resume_name":
                resume_df.loc[
                    resume_index,
                    "filename"
                ],

                "score":
                euclidean_scores[
                    job_index
                ][resume_index]
            })

    return pd.DataFrame(results)

In [16]:
def main():

    # Create DataFrames

    resume_df = create_resume_dataframe(
        "/content/resumes"
    )

    job_df = create_job_dataframe(
        "/content/20_Job_Descriptions (2).pdf"
    )

    # Save CSV

    save_csv(
        resume_df,
        "resumes.csv"
    )

    save_csv(
        job_df,
        "job_descriptions.csv"
    )

    # Embeddings

    job_embeddings = generate_embeddings(
        job_df["text"].tolist()
    )

    resume_embeddings = generate_embeddings(
        resume_df["text"].tolist()
    )

    # Save Embeddings

    save_embeddings(
        job_embeddings,
        "job_embeddings.npy"
    )

    save_embeddings(
        resume_embeddings,
        "resume_embeddings.npy"
    )

    # Similarities

    cosine_scores, euclidean_scores = (
        calculate_scores(
            job_embeddings,
            resume_embeddings
        )
    )

    # Matches

    match_df = get_top_matches(
        job_df,
        resume_df,
        cosine_scores,
        euclidean_scores,
        top_k=10
    )

    save_csv(
        match_df,
        "job_resume_top10_matches.csv"
    )

    print(
        match_df.head()
    )

In [17]:
main()

Saved: resumes.csv
Saved: job_descriptions.csv


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

KeyError: 'text'

In [25]:
resume_df = create_resume_dataframe(
    "/content/resumes"
)

In [23]:
import os

print(os.path.getsize("/content/resumes.csv"))

4


In [26]:
print(resume_df.shape)

resume_df.head()

(0, 0)


""


In [27]:
print(resume_df.shape)

resume_df.head()

(0, 0)


""


In [28]:
import os

print(os.path.exists("/content/resumes"))

True


In [29]:
import os

print(os.listdir("/content/resumes"))

['Education']


In [30]:
from pathlib import Path

pdfs = list(Path("/content/resumes").rglob("*.pdf"))

print("Total PDFs:", len(pdfs))

for pdf in pdfs[:10]:
    print(pdf)

Total PDFs: 5000
/content/resumes/Education/Resume_034753_Dawn_Gardner.pdf
/content/resumes/Education/Resume_034104_Brian_Mack.pdf
/content/resumes/Education/Resume_033269_Kevin_Lin.pdf
/content/resumes/Education/Resume_033860_Justin_Castaneda.pdf
/content/resumes/Education/Resume_033923_Jackson_Mcguire.pdf
/content/resumes/Education/Resume_031914_James_Atkinson.pdf
/content/resumes/Education/Resume_032000_Nathan_Skinner.pdf
/content/resumes/Education/Resume_033698_James_Hall.pdf
/content/resumes/Education/Resume_033836_Daniel_Stone.pdf
/content/resumes/Education/Resume_030402_Richard_Gonzales.pdf


In [33]:
from pathlib import Path

folder = Path("/content/resumes")

count = 0

for pdf_file in folder.rglob("*.pdf"):
    count += 1
    print(pdf_file)
    break

print("Total:", count)

/content/resumes/Education/Resume_034753_Dawn_Gardner.pdf
Total: 1


In [34]:
def create_resume_dataframe(folder_path):

    records = []

    folder = Path(folder_path)

    for pdf_file in folder.rglob("*.pdf"):

        try:

            records.append({

                "filename": pdf_file.name,

                "text":
                extract_text_from_pdf(
                    pdf_file
                )
            })

        except Exception as e:

            records.append({

                "filename":
                pdf_file.name,

                "text": "",

                "error":
                str(e)
            })

    return pd.DataFrame(records)

In [35]:
resume_df = create_resume_dataframe(
    "/content/resumes"
)

print(resume_df.shape)

(5000, 2)


In [36]:
job_df = create_job_dataframe(
    "/content/20_Job_Descriptions (2).pdf"
)

print(job_df.shape)

job_df.head()

(20, 2)


,name,text
0,20 Professional Job Descriptions,Azure Data Engineer\nJob Summary\nWe are seeki...
1,Azure DevOps Engineer,Job Summary\nWe are seeking a skilled Azure De...
2,Data Scientist,Job Summary\nWe are seeking a skilled Data Sci...
3,Data Analyst,Job Summary\nWe are seeking a skilled Data Ana...
4,Full Stack Developer,Job Summary\nWe are seeking a skilled Full Sta...


In [37]:
job_embeddings = generate_embeddings(
    job_df["text"].tolist()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [38]:
resume_embeddings = generate_embeddings(
    resume_df["text"].tolist()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [39]:
save_embeddings(
    job_embeddings,
    "job_embeddings.npy"
)

save_embeddings(
    resume_embeddings,
    "resume_embeddings.npy"
)

Saved: job_embeddings.npy
Saved: resume_embeddings.npy


In [40]:
cosine_scores, euclidean_scores = (
    calculate_scores(
        job_embeddings,
        resume_embeddings
    )
)

In [41]:
match_df = get_top_matches(
    job_df,
    resume_df,
    cosine_scores,
    euclidean_scores,
    top_k=10
)

In [42]:
save_csv(
    match_df,
    "job_resume_top10_matches.csv"
)

Saved: job_resume_top10_matches.csv


In [43]:
match_df[
    match_df["job_name"] ==
    "Data Scientist"
]

,job_name,method,rank,resume_name,score
40,Data Scientist,Cosine Similarity,1,Resume_034496_Susan_Watson.pdf,0.616284
41,Data Scientist,Cosine Similarity,2,Resume_033674_Edgar_Watts.pdf,0.597925
42,Data Scientist,Cosine Similarity,3,Resume_034629_Cody_Carter.pdf,0.567568
43,Data Scientist,Cosine Similarity,4,Resume_031631_Todd_Burgess.pdf,0.565577
44,Data Scientist,Cosine Similarity,5,Resume_034893_Pamela_Swanson.pdf,0.561929
45,Data Scientist,Cosine Similarity,6,Resume_033394_Steven_Watkins.pdf,0.557796
46,Data Scientist,Cosine Similarity,7,Resume_033028_Kaylee_Anderson.pdf,0.556962
47,Data Scientist,Cosine Similarity,8,Resume_030485_Alexander_Cain.pdf,0.552722
48,Data Scientist,Cosine Similarity,9,Resume_032481_Christopher_Smith.pdf,0.549528
49,Data Scientist,Cosine Similarity,10,Resume_034361_Brandon_Carter.pdf,0.548916


In [44]:
np.save(
    "resume_embeddings.npy",
    resume_embeddings
)